In [1]:
import pandas as pd
import numpy as np
from pykrx import stock

# =========================
# 설정
# =========================
START_YEAR = 2015
END_YEAR = 2024
TOP_N = 50
INITIAL_CAPITAL = 100_000_000  # 1억원

portfolio_returns = []

# =========================
# 연도별 리밸런싱 백테스트
# =========================
for year in range(START_YEAR, END_YEAR):

    rebalance_date = f"{year}1230"
    start_date = f"{year + 1}0102"
    end_date = f"{year + 1}1230"

    print(f"백테스트 연도: {year + 1}")

    # 1. KOSPI 펀더멘털 데이터
    fund = stock.get_market_fundamental(
        rebalance_date,
        market="KOSPI"
    )

    fund = fund.reset_index()

    if "티커" in fund.columns:
        fund = fund.rename(columns={"티커": "ticker"})
    elif "index" in fund.columns:
        fund = fund.rename(columns={"index": "ticker"})

    # 2. PER, PBR 정리
    data = fund[["ticker", "PER", "PBR"]].copy()

    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna(subset=["PER", "PBR"])

    data = data[
        (data["PER"] > 0) &
        (data["PBR"] > 0)
    ]

    # 3. 저PER + 저PBR 순위
    data["per_rank"] = data["PER"].rank(ascending=True)
    data["pbr_rank"] = data["PBR"].rank(ascending=True)

    data["total_rank"] = data["per_rank"] + data["pbr_rank"]

    # 4. 상위 50개 종목 선정
    selected = data.sort_values("total_rank").head(TOP_N).copy()
    tickers = selected["ticker"].tolist()

    # 동일가중
    weight = 1 / len(tickers)

    # 5. 개별 종목 수익률 계산
    stock_return_list = []

    for ticker in tickers:
        try:
            price = stock.get_market_ohlcv(
                start_date,
                end_date,
                ticker
            )

            if price.empty:
                continue

            close = price["종가"]
            daily_return = close.pct_change().dropna()

            stock_return_list.append(daily_return * weight)

        except Exception as e:
            print(f"{ticker} 데이터 오류: {e}")
            continue

    # 6. 포트폴리오 일간 수익률
    if len(stock_return_list) == 0:
        continue

    yearly_port_return = pd.concat(stock_return_list, axis=1).sum(axis=1)
    portfolio_returns.append(yearly_port_return)

# =========================
# 전체 수익률 계산
# =========================
strategy_return = pd.concat(portfolio_returns).sort_index()

result = pd.DataFrame()
result["daily_return"] = strategy_return
result["cum_return"] = (1 + result["daily_return"]).cumprod()
result["portfolio_value"] = INITIAL_CAPITAL * result["cum_return"]

# =========================
# 성과 지표 계산
# =========================
total_return = result["cum_return"].iloc[-1] - 1

years = len(result) / 252
cagr = result["cum_return"].iloc[-1] ** (1 / years) - 1

volatility = result["daily_return"].std() * np.sqrt(252)

sharpe_ratio = (
    result["daily_return"].mean() /
    result["daily_return"].std()
) * np.sqrt(252)

cum_max = result["cum_return"].cummax()
drawdown = result["cum_return"] / cum_max - 1
mdd = drawdown.min()

# =========================
# 결과 출력
# =========================
print("\n===== 백테스트 결과 =====")
print(f"누적수익률: {total_return:.2%}")
print(f"CAGR: {cagr:.2%}")
print(f"연환산 변동성: {volatility:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"MDD: {mdd:.2%}")

print("\n===== 최종 포트폴리오 가치 =====")
print(f"{result['portfolio_value'].iloc[-1]:,.0f} 원")

C:\Users\JYB\AppData\Local\Programs\Python\Python39\lib\site-packages\pykrx\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


백테스트 연도: 2016


KeyError: "None of [Index(['BPS', 'PER', 'PBR', 'EPS', 'DIV', 'DPS'], dtype='object')] are in the [columns]"

In [2]:
import pandas as pd
import numpy as np
from pykrx import stock

START_YEAR = 2015
END_YEAR = 2024
TOP_N = 50
INITIAL_CAPITAL = 100_000_000


def get_safe_fundamental(date, market="KOSPI", max_retry=20):
    """
    입력 날짜 기준으로 과거 방향으로 이동하면서
    정상적인 PER/PBR 데이터가 나오는 날짜를 찾는다.
    """
    date = pd.to_datetime(date)

    for _ in range(max_retry):
        d = date.strftime("%Y%m%d")

        try:
            fund = stock.get_market_fundamental(d, market=market)

            if fund.empty:
                date -= pd.Timedelta(days=1)
                continue

            required_cols = {"PER", "PBR"}

            if not required_cols.issubset(set(fund.columns)):
                date -= pd.Timedelta(days=1)
                continue

            fund = fund.reset_index()

            if "티커" in fund.columns:
                fund = fund.rename(columns={"티커": "ticker"})
            elif "index" in fund.columns:
                fund = fund.rename(columns={"index": "ticker"})

            return d, fund

        except Exception:
            date -= pd.Timedelta(days=1)

    raise ValueError(f"{date} 근처에서 정상 펀더멘털 데이터를 찾지 못했습니다.")


def get_safe_price(start_date, end_date, ticker):
    price = stock.get_market_ohlcv(start_date, end_date, ticker)

    if price.empty:
        return None

    if "종가" not in price.columns:
        return None

    return price["종가"]


portfolio_returns = []
portfolio_history = []

for year in range(START_YEAR, END_YEAR):

    # 매년 말 기준으로 종목 선정
    target_rebalance_date = f"{year}-12-30"

    # 다음 해 1년 보유
    start_date = f"{year + 1}0102"
    end_date = f"{year + 1}1230"

    print(f"\n리밸런싱 기준연도: {year}")

    rebalance_date, fund = get_safe_fundamental(
        target_rebalance_date,
        market="KOSPI"
    )

    print(f"실제 사용한 펀더멘털 날짜: {rebalance_date}")

    data = fund[["ticker", "PER", "PBR"]].copy()

    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna(subset=["PER", "PBR"])

    data = data[
        (data["PER"] > 0) &
        (data["PBR"] > 0)
    ]

    data["per_rank"] = data["PER"].rank(ascending=True)
    data["pbr_rank"] = data["PBR"].rank(ascending=True)
    data["total_rank"] = data["per_rank"] + data["pbr_rank"]

    selected = data.sort_values("total_rank").head(TOP_N).copy()

    if selected.empty:
        print("선정 종목 없음")
        continue

    selected["weight"] = 1 / len(selected)
    selected["rebalance_year"] = year
    selected["rebalance_date"] = rebalance_date

    portfolio_history.append(selected)

    stock_return_list = []

    for _, row in selected.iterrows():
        ticker = row["ticker"]
        weight = row["weight"]

        try:
            close = get_safe_price(start_date, end_date, ticker)

            if close is None:
                continue

            daily_return = close.pct_change().dropna()
            stock_return_list.append(daily_return * weight)

        except Exception as e:
            print(f"{ticker} 가격 데이터 오류: {e}")
            continue

    if len(stock_return_list) == 0:
        continue

    yearly_return = pd.concat(stock_return_list, axis=1).sum(axis=1)
    portfolio_returns.append(yearly_return)


strategy_return = pd.concat(portfolio_returns).sort_index()

result = pd.DataFrame()
result["daily_return"] = strategy_return
result["cum_return"] = (1 + result["daily_return"]).cumprod()
result["portfolio_value"] = INITIAL_CAPITAL * result["cum_return"]

total_return = result["cum_return"].iloc[-1] - 1
years = len(result) / 252
cagr = result["cum_return"].iloc[-1] ** (1 / years) - 1
volatility = result["daily_return"].std() * np.sqrt(252)
sharpe_ratio = (
    result["daily_return"].mean() /
    result["daily_return"].std()
) * np.sqrt(252)

cum_max = result["cum_return"].cummax()
drawdown = result["cum_return"] / cum_max - 1
mdd = drawdown.min()

print("\n===== 백테스트 결과 =====")
print(f"누적수익률: {total_return:.2%}")
print(f"CAGR: {cagr:.2%}")
print(f"연환산 변동성: {volatility:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"MDD: {mdd:.2%}")
print(f"최종 포트폴리오 가치: {result['portfolio_value'].iloc[-1]:,.0f} 원")


리밸런싱 기준연도: 2015


ValueError: 2015-12-10 00:00:00 근처에서 정상 펀더멘털 데이터를 찾지 못했습니다.

In [3]:
pip install -U pykrx

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 3.4 MB/s eta 0:00:01
   ---------------------------- ----------- 1.6/2.2 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 4.1 MB/s  0:00:00
  Attempting uninstall: pykrx
    Found existing installation: pykrx 1.0.48
    Uninstalling pykrx-1.0.48:
      Successfully uninstalled pykrx-1.0.48
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from pykrx import stock

test = stock.get_market_fundamental_by_ticker("20240102", market="KOSPI")
print(test.head())
print(test.columns)

KeyError: "None of [Index(['BPS', 'PER', 'PBR', 'EPS', 'DIV', 'DPS'], dtype='object')] are in the [columns]"

In [6]:
!pip uninstall pykrx -y
!pip install pykrx==1.2.7

Found existing installation: pykrx 1.0.51
Uninstalling pykrx-1.0.51:
  Successfully uninstalled pykrx-1.0.51


ERROR: Ignored the following versions that require a different python version: 1.1.3.dev0 Requires-Python >=3.10; 1.1.4.dev0 Requires-Python >=3.10; 1.1.6.dev0 Requires-Python >=3.10; 1.2.1.dev0 Requires-Python >=3.10; 1.2.1.post1.dev0 Requires-Python >=3.10; 1.2.3 Requires-Python >=3.10; 1.2.3.dev0 Requires-Python >=3.10; 1.2.4 Requires-Python >=3.10; 1.2.5 Requires-Python >=3.10; 1.2.6 Requires-Python >=3.10; 1.2.7 Requires-Python >=3.10
ERROR: Could not find a version that satisfies the requirement pykrx==1.2.7 (from versions: 0.0.1, 0.0.2, 0.0.3, 0.0.4, 0.0.5, 0.0.6, 0.0.7, 0.0.8, 0.0.9, 0.0.10, 0.1.0, 0.1.1, 0.1.2, 0.1.3, 0.1.4, 0.1.5, 0.1.6, 0.1.7, 0.1.8, 0.1.9, 0.1.10, 0.1.11, 0.1.12, 0.1.13, 0.1.14, 0.1.15, 0.1.16, 0.1.17, 0.1.18, 0.1.19, 0.1.20, 0.1.21, 0.1.22, 0.1.23, 0.1.25, 0.1.26, 0.1.27, 0.1.28, 0.1.29, 0.1.30, 0.1.31, 0.1.32, 0.1.33, 0.1.34, 0.1.35, 0.1.36, 0.1.37, 0.1.38, 0.1.39, 0.1.40, 0.1.41, 0.1.42, 1.0.0, 1.0.1, 1.0.2, 1.0.3, 1.0.4, 1.0.6, 1.0.7, 1.0.8, 1.0.9, 1.0.

In [7]:
import pykrx
from pykrx import stock

print(pykrx.__version__)

df = stock.get_market_ohlcv_by_ticker("20240102", market="KOSPI")
print(df.head())
print(df.columns)

1.0.48


KeyError: "None of [Index(['시가', '고가', '저가', '종가'], dtype='object')] are in the [columns]"